# Model Comparison: Ridge vs XGBoost vs LightGBM

Forecast target: `Avg Transactions per Agency`  
Workflow: hash dataset -> make time-series features -> chronological train/test/eval split -> train three pipelines -> choose winner -> generate model card.

This is intentionally compact and production-like without overengineering.

## 0. Local setup

Run the install cell below once if your local environment does not already have the packages.


In [ ]:
# Uncomment and run once if needed:
# %pip install pandas openpyxl scikit-learn xgboost lightgbm joblib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## 1. Config

In [ ]:
# Put the Excel file in the same folder as this notebook, or in a ./data folder.
DATA_FILE_NAME = "LEB3443M022026-range - Number of Real Estate Transactions - augmented-1000-rows.xlsx"

FILE_PATH = Path(DATA_FILE_NAME)
if not FILE_PATH.exists():
    FILE_PATH = Path("data") / DATA_FILE_NAME

if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE_NAME}. Put it next to the notebook, "
        "or place it inside a ./data folder, or set FILE_PATH manually."
    )

SHEET_NAME = "3443_augmented"
DATE_COL = "Month"

# Change to "Transactions" if you want total monthly transactions instead.
TARGET_COL = "Avg Transactions per Agency"

RANDOM_STATE = 42
LAGS = [1, 2, 3, 6, 12]
ROLLING_WINDOWS = [3, 6, 12]

OUT_DIR = Path("model_comparison_artifacts")
OUT_DIR.mkdir(exist_ok=True)


## 2. Helper functions

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def metrics_dict(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape_pct": float(safe_mape(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def load_monthly_data(path: Path) -> pd.DataFrame:
    # Actual header row in this workbook is Excel row 7, hence header=6.
    df = pd.read_excel(path, sheet_name=SHEET_NAME, header=6, usecols="A:D")
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(how="all")

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    for col in df.columns:
        if col != DATE_COL:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL)
    df = df.drop_duplicates(subset=[DATE_COL], keep="last")

    # Monthly start frequency.
    return df.set_index(DATE_COL).asfreq("MS")


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Date/time features
    out["year"] = out.index.year
    out["month"] = out.index.month
    out["quarter"] = out.index.quarter
    out["trend"] = np.arange(len(out))

    # Seasonality features
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)

    # Lag features
    for lag in LAGS:
        out[f"{TARGET_COL}_lag_{lag}"] = out[TARGET_COL].shift(lag)

    # Rolling features using only past values
    shifted = out[TARGET_COL].shift(1)
    for window in ROLLING_WINDOWS:
        out[f"{TARGET_COL}_roll_mean_{window}"] = shifted.rolling(window).mean()
        out[f"{TARGET_COL}_roll_std_{window}"] = shifted.rolling(window).std()
        out[f"{TARGET_COL}_roll_min_{window}"] = shifted.rolling(window).min()
        out[f"{TARGET_COL}_roll_max_{window}"] = shifted.rolling(window).max()

    return out


def chronological_split(X, y, train_frac=0.70, test_frac=0.15):
    n = len(X)
    train_end = int(n * train_frac)
    test_end = int(n * (train_frac + test_frac))

    return (
        X.iloc[:train_end], y.iloc[:train_end],
        X.iloc[train_end:test_end], y.iloc[train_end:test_end],
        X.iloc[test_end:], y.iloc[test_end:]
    )

## 3. Load data and hash the dataset

In [ ]:
dataset_hash = sha256_file(FILE_PATH)
monthly = load_monthly_data(FILE_PATH)

print("Dataset SHA-256:", dataset_hash)
print("Shape:", monthly.shape)
print("Period:", monthly.index.min(), "to", monthly.index.max())
print("Missing values:\n", monthly.isna().sum())
monthly.head()

## 4. Feature engineering and chronological split

Same-month raw columns are dropped to avoid leakage. The models use lag, rolling, trend, and seasonality features.

In [ ]:
feature_df = make_features(monthly)

# Avoid leakage from same-month raw columns like Transactions and Agency Listings.
raw_same_month_cols = [c for c in monthly.columns if c != TARGET_COL]

X = feature_df.drop(columns=[TARGET_COL] + raw_same_month_cols, errors="ignore")
y = feature_df[TARGET_COL]

valid = y.notna()
X, y = X.loc[valid], y.loc[valid]

feature_cols = X.columns.tolist()

X_train, y_train, X_test, y_test, X_eval, y_eval = chronological_split(X, y)

print("Train:", X_train.index.min().date(), "to", X_train.index.max().date(), X_train.shape)
print("Test: ", X_test.index.min().date(), "to", X_test.index.max().date(), X_test.shape)
print("Eval: ", X_eval.index.min().date(), "to", X_eval.index.max().date(), X_eval.shape)

X.head()

## 5. Define one sklearn Pipeline per model

In [ ]:
numeric_impute_only = ColumnTransformer(
    [("num", SimpleImputer(strategy="median"), feature_cols)],
    remainder="drop"
)

numeric_impute_scale = ColumnTransformer(
    [("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), feature_cols)],
    remainder="drop"
)

models = {
    "ridge": Pipeline([
        ("preprocess", numeric_impute_scale),
        ("model", Ridge(alpha=10.0))
    ]),

    "xgboost": Pipeline([
        ("preprocess", numeric_impute_only),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            n_estimators=350,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]),

    "lightgbm": Pipeline([
        ("preprocess", numeric_impute_only),
        ("model", LGBMRegressor(
            objective="regression",
            n_estimators=350,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=-1
        ))
    ])
}

## 6. Train, test, evaluate, and select the winner

In [ ]:
records = []
pred_frames = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)

    for split_name, X_split, y_split in [
        ("train", X_train, y_train),
        ("test", X_test, y_test),
        ("eval", X_eval, y_eval),
    ]:
        pred = pipe.predict(X_split)
        records.append({"model": name, "split": split_name, **metrics_dict(y_split, pred)})

        pred_frames.append(pd.DataFrame({
            "date": X_split.index,
            "model": name,
            "split": split_name,
            "actual": y_split.values,
            "prediction": pred
        }))

metrics_df = pd.DataFrame(records)
preds_df = pd.concat(pred_frames, ignore_index=True)

winner_row = metrics_df[metrics_df["split"] == "eval"].sort_values("mae").iloc[0]
winner_name = winner_row["model"]
winner_model = models[winner_name]

print("Winner:", winner_name)
print("Winner eval MAE:", winner_row["mae"])

metrics_df.sort_values(["split", "mae"])

## 7. Save winner and write model card

In [ ]:
model_path = OUT_DIR / f"{winner_name}_winner_model.joblib"
metrics_path = OUT_DIR / "model_metrics.csv"
preds_path = OUT_DIR / "model_predictions.csv"
model_card_path = OUT_DIR / "model_card.md"

bundle = {
    "model": winner_model,
    "winner_name": winner_name,
    "feature_cols": feature_cols,
    "target_col": TARGET_COL,
    "date_col": DATE_COL,
    "lags": LAGS,
    "rolling_windows": ROLLING_WINDOWS,
    "dataset_sha256": dataset_hash,
    "metrics": metrics_df.to_dict(orient="records"),
}

joblib.dump(bundle, model_path)
metrics_df.to_csv(metrics_path, index=False)
preds_df.to_csv(preds_path, index=False)

periods = {
    "full": (str(monthly.index.min().date()), str(monthly.index.max().date()), len(monthly)),
    "train": (str(X_train.index.min().date()), str(X_train.index.max().date()), len(X_train)),
    "test": (str(X_test.index.min().date()), str(X_test.index.max().date()), len(X_test)),
    "eval": (str(X_eval.index.min().date()), str(X_eval.index.max().date()), len(X_eval)),
}

metrics_md = metrics_df.pivot(index="model", columns="split", values="mae").sort_values("eval").round(4).to_markdown()
all_metrics_md = metrics_df.sort_values(["split", "mae"]).round(4).to_markdown(index=False)

model_card = f"""# Model Card: Monthly Average Transactions Forecaster

## Model overview
- **Selected model:** `{winner_name}`
- **Task:** Forecast `{TARGET_COL}` at monthly frequency.
- **Modeling approach:** Supervised time-series regression using calendar, trend, lag, and rolling-window features.
- **Selection rule:** Lowest MAE on the chronological eval split.
- **Created at:** {datetime.now(timezone.utc).isoformat()}

## Dataset
- **Source file:** `{FILE_PATH.name}`
- **Sheet:** `{SHEET_NAME}`
- **Dataset SHA-256:** `{dataset_hash}`
- **Full period:** {periods['full'][0]} to {periods['full'][1]}
- **Rows:** {periods['full'][2]}
- **Target:** `{TARGET_COL}`

## Split strategy
Chronological split, no shuffling.

| Split | Period | Rows |
|---|---:|---:|
| Train | {periods['train'][0]} to {periods['train'][1]} | {periods['train'][2]} |
| Test | {periods['test'][0]} to {periods['test'][1]} | {periods['test'][2]} |
| Eval | {periods['eval'][0]} to {periods['eval'][1]} | {periods['eval'][2]} |

## Features
- Calendar features: year, month, quarter
- Trend feature: sequential month index
- Seasonality features: month sine/cosine
- Lag features: {LAGS}
- Rolling windows: {ROLLING_WINDOWS}
- Same-month raw columns are excluded to avoid leakage.

## Model comparison by MAE
{metrics_md}

## Full metrics
{all_metrics_md}

## Intended use
Use for monthly forecasting of `{TARGET_COL}` when historical monthly observations are available and the future period is similar to the historical/synthetic training distribution.

## Limitations and risks
- The file appears to contain augmented/synthetic future rows through 2094, so real-world generalization should be validated on non-augmented historical data.
- Recursive multi-month forecasts can accumulate error.
- The model does not include macroeconomic drivers, policy changes, season shocks, or known future events.
- Same-month transaction/listing columns were intentionally excluded to avoid leakage.

## Production notes
- Store the raw file hash with every training run.
- Recompute metrics after each retrain.
- Monitor prediction drift and MAE over recent months.
- Refit only after confirming eval performance is stable on real non-synthetic data.

## Saved artifacts
- Model bundle: `{model_path.name}`
- Metrics: `{metrics_path.name}`
- Predictions: `{preds_path.name}`
"""

model_card_path.write_text(model_card, encoding="utf-8")

print("Saved model:", model_path)
print("Saved metrics:", metrics_path)
print("Saved predictions:", preds_path)
print("Saved model card:", model_card_path)

## 8. Show model card

In [ ]:
print(model_card)